# DJR WGS sex versus field sex

Compare WGS/Y-based sex assignments with field metadata. If WGS sex is missing, retain field sex.

In [1]:
library(data.table)

wgs_file <- "/scratch/ztobias/BC_stickle_expanded_WGS/metadata/derived/wgs_sex_assignment_reconciliation_DJR.tsv"
rna24_file <- "/scratch/ztobias/Gacu_BC_txmics/metadata/2024_BC_RNA_extractions.csv"
meta25_file <- "/scratch/ztobias/Gacu_BC_txmics/metadata/BC Spring 2025 Collections and Molecular Metadata Sheet 1.csv"

wgs <- fread(wgs_file)
rna24 <- fread(rna24_file, fill = TRUE)
setnames(rna24, 1, "sample")
meta25 <- fread(meta25_file, sep = ",", header = TRUE, fill = TRUE,
  encoding = "UTF-8")

norm_sex <- function(x) {
  x <- tolower(trimws(as.character(x)))
  fifelse(grepl("^f|female", x), "female",
    fifelse(grepl("^m|male", x), "male", NA_character_))
}

wgs <- wgs[grepl("^(2024|2025)_DJR_", sample)]
wgs[, wgs_sex := norm_sex(assigned_sex_y_djr)]
wgs <- wgs[, .(sample, wgs_sex, y_missing_proportion,
  assignment_margin_djr, assignment_confident_djr)]

field24 <- rna24[grepl("^2024_DJR_", sample), .(
  sample, field_sex_raw = as.character(sex), field_sex = norm_sex(sex),
  field_notes = as.character(notes)
)]

setnames(meta25, names(meta25)[1], "sample")
sex_col <- match("Sex", names(meta25))
eggs_col <- match("Eggs?", names(meta25))
notes_col <- match("Field notes", names(meta25))
stained_sex_col <- match("stained sex check", names(meta25))
stopifnot(!is.na(sex_col), !is.na(eggs_col), !is.na(notes_col), !is.na(stained_sex_col))
field25 <- meta25[grepl("^(2024|2025)_DJR_", sample), .(
  sample,
  field_sex_raw = as.character(get(names(meta25)[sex_col])),
  field_sex = norm_sex(get(names(meta25)[sex_col])),
  field_notes = paste(get(names(meta25)[notes_col]), get(names(meta25)[eggs_col]),
    get(names(meta25)[stained_sex_col]), sep = " | ")
)]

field <- rbindlist(list(field24, field25), fill = TRUE)
field <- field[!duplicated(sample)]

sex_compare <- merge(field, wgs, by = "sample", all = TRUE)
sex_compare[, mismatch := !is.na(field_sex) & !is.na(wgs_sex) & field_sex != wgs_sex]
sex_compare[, final_sex := fifelse(!is.na(wgs_sex), wgs_sex, field_sex)]
sex_compare[, final_sex_source := fifelse(!is.na(wgs_sex), "WGS",
  fifelse(!is.na(field_sex), "field", NA_character_))]

cat("Mismatches:\n")
print(sex_compare[mismatch == TRUE])
cat("\nField-sex fallback samples:\n")
print(sex_compare[is.na(wgs_sex) & !is.na(field_sex)])

Warning message:
“package ‘data.table’ was built under R version 4.1.3”


Mismatches:
          sample field_sex field_notes wgs_sex y_missing_proportion
 1: 2024_DJR_041      male              female            0.6174170
 2: 2024_DJR_042    female                male            0.0666968
 3: 2024_DJR_108      male              female            0.6243880
 4: 2024_DJR_138      male              female            0.6317470
 5: 2024_DJR_140      male              female            0.6271320
 6: 2024_DJR_155      male              female            0.6249120
 7: 2024_DJR_156    female                male            0.0820902
 8: 2024_DJR_158      male              female            0.6248350
 9: 2024_DJR_159      male              female            0.6276590
10: 2024_DJR_168    female                male            0.0778647
11: 2024_DJR_169      male              female            0.6394090
12: 2024_DJR_178    female                male            0.0783947
13: 2024_DJR_181      male              female            0.6289970
14: 2024_DJR_186      male          

In [2]:
out_dir <- "/scratch/ztobias/Gacu_BC_txmics/metadata/derived"
fwrite(sex_compare, file.path(out_dir, "DJR_sex_field_comparison.tsv"), sep = "\t")
fwrite(sex_compare[mismatch == TRUE], file.path(out_dir, "DJR_sex_mismatches.tsv"), sep = "\t")

sex_compare[, .(n = .N), by = .(final_sex_source, final_sex)]

final_sex_source,final_sex,n
<chr>,<chr>,<int>
WGS,male,306
WGS,female,348
NA,NA,582
field,female,3
field,male,1


In [4]:
# BRB-seq samples: retain fish with a gill and/or liver barcode.
brb <- fread("/scratch/ztobias/Gacu_BC_txmics/metadata/brb_sample_barcodes.tsv")

setnames(brb, "sample_id", "sample")
brb <- brb[grepl("^(2024|2025)_DJR_", sample)]
brb[, tissue := tolower(tissue)]

brb_presence <- unique(brb[, .(sample, tissue)])
brb_presence[, present := TRUE]

brb_presence <- dcast(
  brb_presence,
  sample ~ tissue,
  value.var = "present",
  fill = FALSE
)

if (!"gill" %in% names(brb_presence))
  brb_presence[, gill := FALSE]

if (!"liver" %in% names(brb_presence))
  brb_presence[, liver := FALSE]

setnames(
  brb_presence,
  c("gill", "liver"),
  c("brb_gill", "brb_liver")
)

brb_sex <- merge(
  brb_presence,
  sex_compare,
  by = "sample",
  all.x = TRUE
)

brb_sex[, brb_tissues := fifelse(
  brb_gill & brb_liver,
  "gill+liver",
  fifelse(brb_gill, "gill", "liver")
)]

brb_sex <- brb_sex[order(sample)]

brb_sex <- brb_sex[, .(
  sample,
  brb_tissues,
  brb_gill,
  brb_liver,
  field_sex_raw,
  field_sex,
  wgs_sex,
  final_sex,
  final_sex_source,
  mismatch,
  y_missing_proportion,
  assignment_confident_djr,
  field_notes
)]

cat("BRB-seq samples by tissue:\n")
print(brb_sex[, .(n = .N), by = brb_tissues])

cat("BRB-seq samples without WGS sex:\n")
print(
  brb_sex[
    is.na(wgs_sex),
    .(sample, brb_tissues, field_sex_raw, field_sex, final_sex, field_notes)
  ]
)

fwrite(
  brb_sex,
  file.path(out_dir, "DJR_sex_BRB_samples.tsv"),
  sep = "\t"
)

brb_sex

BRB-seq samples by tissue:
   brb_tissues   n
1:       liver  17
2:  gill+liver 622
3:        gill  14
BRB-seq samples without WGS sex:
         sample brb_tissues field_sex final_sex  field_notes
1: 2024_DJR_196  gill+liver      <NA>      <NA> not included
2: 2025_DJR_177        gill      <NA>      <NA>         <NA>
3: 2025_DJR_193        gill      <NA>      <NA>         <NA>
4: 2025_DJR_195        gill      <NA>      <NA>         <NA>
5: 2025_DJR_196        gill      <NA>      <NA>         <NA>
6: 2025_DJR_272        gill      <NA>      <NA>         <NA>


sample,brb_tissues,brb_gill,brb_liver,field_sex,wgs_sex,final_sex,final_sex_source,mismatch,y_missing_proportion,assignment_confident_djr,field_notes
<chr>,<chr>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<lgl>,<dbl>,<lgl>,<chr>
2024_DJR_001,liver,FALSE,TRUE,male,male,male,WGS,FALSE,0.0883885,TRUE,
2024_DJR_002,gill+liver,TRUE,TRUE,female,female,female,WGS,FALSE,0.6170590,TRUE,
2024_DJR_003,gill+liver,TRUE,TRUE,female,female,female,WGS,FALSE,0.6039790,TRUE,
2024_DJR_004,gill+liver,TRUE,TRUE,male,male,male,WGS,FALSE,0.0782662,TRUE,
2024_DJR_005,gill,TRUE,FALSE,female,female,female,WGS,FALSE,0.6244400,TRUE,
2024_DJR_006,gill+liver,TRUE,TRUE,male,male,male,WGS,FALSE,0.0747598,TRUE,
2024_DJR_007,gill+liver,TRUE,TRUE,female,female,female,WGS,FALSE,0.6161770,TRUE,
2024_DJR_008,gill+liver,TRUE,TRUE,male,male,male,WGS,FALSE,0.0775030,TRUE,
2024_DJR_009,gill+liver,TRUE,TRUE,female,female,female,WGS,FALSE,0.6123070,TRUE,
